# RAGNav quickstart (offline hybrid retrieval)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irfanalidv/RAGNav/blob/main/cookbook/ragnav_quickstart.ipynb)

Hybrid **BM25 + sentence-transformers** — no API key. Builds a small index over SQuAD passages and shows **confidence** and **QueryFallback** (retry with a rephrased query when the first pass is uncertain).

In [ ]:
%pip install -q "ragnav[embeddings]>=0.3.0" "datasets>=2.14.0"

In [ ]:
from datasets import load_dataset

from ragnav import RAGNavIndex, RAGNavRetriever
from ragnav.ingest.markdown import ingest_markdown_string

# 50 unique passages from SQuAD validation (order-stable dedupe)
ds = load_dataset("rajpurkar/squad", split="validation[:200]")
seen: set[str] = set()
contexts: list[str] = []
for row in ds:
    c = row["context"]
    if c in seen:
        continue
    seen.add(c)
    contexts.append(c)
    if len(contexts) >= 50:
        break

documents = []
blocks: list = []
for i, ctx in enumerate(contexts):
    doc, doc_blocks = ingest_markdown_string(ctx, name=f"passage_{i}.md")
    documents.append(doc)
    blocks.extend(doc_blocks)

index = RAGNavIndex.build(
    documents=documents,
    blocks=blocks,
    use_sentence_transformers=True,
    vector_model="all-MiniLM-L6-v2",
    embed_batch_size=32,
)
print(f"Index ready: {len(index.blocks_by_id)} blocks from {len(documents)} documents")

In [ ]:
retriever = RAGNavRetriever(index=index)

questions = [
    "What is the capital of France?",
    "Who invented the telephone?",
    "When did World War II end?",
]

for q in questions:
    result = retriever.retrieve(
        q,
        top_k=3,
        expand_structure=False,
        expand_graph=False,
    )
    top = result.blocks[0].text[:160].replace("\n", " ") if result.blocks else "(no result)"
    print(f"Q: {q}")
    print(f"Top snippet: {top}...")
    print(f"Confidence: {result.confidence.value}")
    print()

## QueryFallback — automatic retry on low / medium confidence

`FakeLLMClient` echoes prompts, so it is a poor query rewriter. For this demo we use a tiny stub that returns **one** alternative line when the fallback asks for variations. In production, plug in any `LLMClient` (e.g. Mistral with `pip install ragnav[mistral]`).

In [ ]:
from ragnav.retrieval import QueryFallback


class DemoVariationLLM:
    """Colab-only: return a single rephrasing line for the variation prompt."""

    def chat(self, *, messages, model=None, temperature=0.7):
        return "When was the Great Wall of China built?\n"


fallback = QueryFallback(retriever=retriever, llm=DemoVariationLLM())

# Deliberately vague wording so the first pass is often weaker than the rephrase
out = fallback.retrieve(
    "Great Wall construction date",
    top_k=5,
    expand_structure=False,
    expand_graph=False,
)
print("queries_tried:", out.queries_tried)
print("improved:", out.improved)
print("final confidence:", out.final_result.confidence.value)
if out.final_result.blocks:
    print("top block:", out.final_result.blocks[0].text[:200].replace("\n", " "))